# Alert 생성기 실험 ③ — 모델 없이 패턴 룰만 vs 이진분류만

같은 원장에서 **세탁 거래를 고르는 일**을 두 가지 방법으로 하고 정답 라벨로 잰다. 묶기(Alert 생성기)는 이 실험에 없다.

| 방법 | 고르는 기준 | 쓰는 것 |
|---|---|---|
| 룰만 | 패턴 6종을 룰로 적어 원장 전체에 건다. 룰에 걸린 근거 거래가 "고른 거래"다 | 거래 시각, 계좌, 금액. **모델 점수는 쓰지 않는다** |
| 이진분류만 | 점수 ≥ 의심 기준 | 이진 모델의 점수. HI-Medium 은 서비스 이진 모델(00번이 만든 점수) |

**룰 하나는 조건 여러 개를 AND 로 묶은 것이다.** 구성(거래 필터 + 집계값을 임계값과 비교 + AND)은 Oracle 시나리오를 따랐고, 조건의 종류와 값은 HI-Small 을 보고 정했다.

| 조건 | 뜻 | 후보 |
|---|---|---|
| ① 거래 필터 | 룰이 세는 거래를 먼저 거른다: 건당 최소 금액, 처음 거래하는 상대만, 환전 직후 송금 | 0절 `FILTERS` |
| ② 기간 | 트리거 시각에서 돌아보는 길이 | 0절 `WINDOWS` |
| ③ 세기 ≥ 조건값 | 기간 안에서 센 패턴의 크기(계좌 수, 경로 길이) | 0절 `GRID` |
| ④ 반복 없음 (FAN 만) | 기간 안에서 모든 상대와 한 번씩만 거래 | 허용 / 없음 |

- 룰 6종: FAN-OUT, FAN-IN, GATHER-SCATTER, SCATTER-GATHER, CYCLE, RANDOM. 정의는 4절과 `rule_lib.py` 에 있다.
- **무패턴 · STACK · BIPARTITE 세탁은 지표에서 뺀다**(0절 `EXCLUDE`). 정답도 오답도 아닌 것으로 둔다: 룰이나 모델이 이 거래를 골라도 맞힌 것으로도 틀린 것으로도 세지 않고, 재현율 분모에도 넣지 않는다. 룰은 원장 전체를 그대로 본다(이 거래들도 상대 계좌로 센다).
- 조건은 미리 정하지 않는다. 후보를 전부 돌린 뒤 **보정 구간에서 고르고**(6절) **평가 구간에서 잰다**(7절). 모델이 검증 구간에서 의심 기준을 고르고 테스트 구간에서 재는 것과 같은 순서다.
- 룰은 패턴이 **완성된 뒤에야** 걸린다. 기본 설정(`LOOKAHEAD_DAYS = 0`)에서는 룰도 모델처럼 평가 구간 끝까지의 거래만 보므로, 구간 끝에 걸친 시도는 룰이 불리하다(1절에서 그런 시도 수를 센다).
- **평가 구간은 모델 점수가 있는 구간과 같다.** HI-Medium 은 00번이 정한 구간(데이터 첫날 + 7일 ~ 09-16)이다. 그 앞 거래는 룰이 돌아보는 이력으로만 쓴다(읽기는 전부 읽는다). 룰도 데이터가 시작한 직후에는 오탐이 많다: 돌아볼 기간과 "처음 거래하는 상대" 이력이 아직 없어서다.
- `MODEL = None` 이거나 점수 파일이 없으면 룰 쪽만 돈다. 8절과 9절의 모델 줄은 빠진다.
- 룰 SQL 은 HI-Small 에서 같은 정의를 Python 반복문으로 짠 계산과 결과가 같은 것을 확인했다(2026-11-23).
- **HI-Large 는 아직 한 번도 안 돌려 봤다.** 소요 시간과 메모리를 모른다. 0절 끝의 "HI-Large 로 돌릴 때" 를 먼저 읽는다.

## 0. 설정

In [ ]:
DATASET = 'HI-Medium'            # 'HI-Medium' / 'HI-Small' / 'HI-Large'. 바꾸면 아래에서 데이터셋별로 적어 둔 값(점수 파일, 구간, 기간 후보, DuckDB 설정)이 같이 바뀐다

import json
from pathlib import Path
import gen_lib as GL

# 데이터셋마다 쓰는 점수 파일: (폴더, 모델 이름). 01번과 같다
SOURCES = {
    'HI-Medium': ('../../parquets/HI-Medium/service_scores', 'multi280_hgb36_v1'),          # 서비스 이진 모델. 00_service_scores.ipynb 가 만든다
    'HI-Small':  ('../binary_classification/results/HI-Small/model_compare', 'trans_s1'),   # 21_model_compare
    'HI-Large':  ('../binary_classification/results/HI-Large/gnn_structure', 'rev_port'),   # 09_gnn_structure
}
EXP, MODEL = Path(SOURCES[DATASET][0]), SOURCES[DATASET][1]      # MODEL = None 으로 바꾸면 룰 쪽만 돈다
print(sorted(p.name for p in EXP.glob('*_test_scores.parquet')))

In [ ]:
# 점수 파일이 아직 없으면 룰 쪽만 돈다 (8절과 9절의 모델 줄이 빠진다)
if MODEL is not None and not (EXP / f'{MODEL}_test_scores.parquet').exists():
    print(f'점수 파일이 없다: {EXP / (MODEL + "_test_scores.parquet")} → 룰 쪽만 돈다')
    MODEL = None
S = GL.score_settings(EXP, MODEL) if MODEL else dict(scores=None, threshold=None, eval_window=None)
SCORES, THRESHOLD = S['scores'], S['threshold']      # 의심 기준. 서비스 모델은 서비스 값, 나머지는 그 모델이 검증 구간에서 고른 값
print(SCORES, THRESHOLD, S['eval_window'])

In [ ]:
# --- 구간
# 평가 구간 = 모델 점수가 있는 구간. 점수 파일에 적혀 있으면 그 구간(00번이 정한 것), 없으면 아래 값 (01번과 같은 방식)
TEST_WINDOW  = tuple(S['eval_window'] or {'HI-Medium': ('2022-09-08', '2022-09-16'), 'HI-Small': ('2022-09-09', '2022-09-10'),
                                          'HI-Large': ('2022-10-17', '2022-10-30')}[DATASET])
# 보정 구간 = 평가 구간 앞. 룰 조건을 고르는 데만 쓴다. HI-Medium 은 첫 4일(이력이 덜 쌓인 날)을 보정에서도 뺐다. 아래 설명 참고
CALIB_WINDOW = {'HI-Medium': ('2022-09-05', '2022-09-07'), 'HI-Small': ('2022-09-01', '2022-09-08'), 'HI-Large': ('2022-10-03', '2022-10-16')}[DATASET]
assert CALIB_WINDOW[1] < TEST_WINDOW[0], '보정 구간은 평가 구간보다 앞이어야 한다'
PAD_DAYS = {'HI-Medium': 14, 'HI-Small': 14, 'HI-Large': 30}[DATASET]     # 보정 구간 앞으로 더 읽는 날짜 수. 룰이 돌아볼 거래다 (가장 긴 기간 이상). 데이터가 그보다 짧으면 처음부터 읽는다
LOOKAHEAD_DAYS = 0               # 평가 구간 뒤로 룰이 더 보는 날짜 수. 0 = 모델과 같은 정보(평가 구간 끝까지). 아래 설명 참고

# --- 지표에서 빼는 세탁. 정답도 오답도 아닌 것으로 두고 뺀다 (이진분류 08 · 09 보고서에서 무패턴을 뺀 것과 같은 계산). 빼지 않으려면 []
EXCLUDE = ['NO_PATTERN', 'STACK', 'BIPARTITE']     # 무패턴(패턴 파일에 없는 세탁), STACK, BIPARTITE

# --- ① 거래 필터 후보. 이름: (건당 최소 금액 USD, 처음 거래하는 상대만 셀지[, 환전 조건을 쓰는 방식 'and' / 'or'])
FILTERS = {
    '없음':        (0, False),
    '1천↑':        (1000, False),
    '첫거래':      (0, True),
    '첫거래+1천↑': (1000, True),
    '첫거래+5천↑': (5000, True),
    # 환전 조건을 AND 로: 환전 직후 송금만 센다
    '환전':              (0, False, 'and'),
    '환전+1천↑':         (1000, False, 'and'),
    '첫거래+환전':       (0, True, 'and'),
    '첫거래+환전+1천↑':  (1000, True, 'and'),
    '첫거래+환전+5천↑':  (5000, True, 'and'),
    # 환전 조건을 OR 로: 첫 거래이거나 환전 직후 송금이면 센다 (금액 조건은 AND)
    '(첫거래 또는 환전)':        (0, True, 'or'),
    '(첫거래 또는 환전)+1천↑':   (1000, True, 'or'),
    '(첫거래 또는 환전)+5천↑':   (5000, True, 'or'),
}
# --- ② 기간 후보(분). 룰엔진 설계안의 기간 목록에 있는 값에서 고른다. HI-Small · HI-Medium 1 · 3 · 7일, HI-Large 7 · 14 · 30일 (아래 설명)
WINDOWS = {'HI-Medium': [1440, 4320, 10080], 'HI-Small': [1440, 4320, 10080], 'HI-Large': [10080, 20160, 43200]}[DATASET]
PATH_WINDOWS = [W for W in WINDOWS if W <= 20160]     # 경로형(CYCLE · RANDOM)은 14일까지만. 설계안 4.4.3 의 상한과 같다
# --- ③ 조건값 후보. 세기가 이 값 이상이면 적중
GRID = {
    'fan_out':        [2, 3, 4, 5, 6, 8, 10, 12, 14, 16],    # 서로 다른 수취 계좌 수
    'fan_in':         [2, 3, 4, 5, 6, 8, 10, 12, 14, 16],    # 서로 다른 송금 계좌 수
    'gather_scatter': [2, 3, 4, 5, 6, 8, 10, 12],            # 송금 계좌 수와 수취 계좌 수 (둘 다)
    'scatter_gather': [2, 3, 4, 5, 6, 8, 10, 12],            # 경유 계좌 수
    'cycle':          [2, 3, 4, 5, 6, 8, 10],                # 순환의 거래 수
    'random':         [2, 3, 4, 5, 6, 7, 8, 10],             # 경로의 거래 수
}
# ④ 반복 없음(FAN 만)은 후보가 '허용' / '없음' 둘이라 따로 적지 않는다

RATIO_MIN, RATIO_MAX = 0.8, 1.25 # 경로형(CYCLE·RANDOM): 이어지는 두 거래의 금액 비(뒤 ÷ 앞, USD). 생성기 정책(block-pin-3)의 금액 띠와 같은 값
FX_RATIO = (RATIO_MIN, RATIO_MAX)  # 환전 직후 송금: 보낸 금액 ÷ 환전해 받은 금액의 범위. 경로형의 금액 비와 같은 값을 쓴다 (아래 설명)
MAX_HOPS = 12                    # 경로형: 따라가는 최대 거래 수
MAX_WALKS = 50_000_000           # 경로형: 경로 수 상한(4-4절의 표)이 이보다 크면 그 (필터, 기간)은 경로를 만들지 않고 건너뛴다
BUSY_MAX, BUSY_MINUTES = 512, 4320   # 바쁜 계좌: 72시간 안 같은 방향 거래가 512건 초과. 생성기 허브 기준과 같은 숫자. BUSY_MAX = None 이면 끈다
PICK_BY = 'F1'                   # 6절에서 조건을 고르는 기준 열

THREADS, MEMORY = {'HI-Medium': (8, '24GB'), 'HI-Small': (4, '8GB'), 'HI-Large': (8, '24GB')}[DATASET]     # DuckDB 설정. 서버 사양에 맞춰 고친다
FILE_DB = DATASET != 'HI-Small'  # True 면 룰 표를 메모리가 아니라 파일(parquets/<DATASET>/rule_engine_work.duckdb)에 둔다. 큰 데이터는 메모리에 안 들어간다
REUSE = True                    # True 면 이미 만든 룰 표는 건너뛴다. 중간에 끊긴 실행을 이어 돌릴 때만 켠다 (FILE_DB 일 때만). 설정을 바꿨으면 False

**거래 필터에 대해.**
- `건당 최소 금액`: 이보다 작은 거래는 룰이 세지 않고 근거 거래도 되지 않는다.
- `처음 거래하는 상대만`: 그 (송금 계좌 → 수취 계좌) 쌍의 **첫 거래**만 센다. 데이터 처음부터 본다. 정상 계좌는 늘 거래하던 상대와 반복해서 거래하므로, 이 필터를 걸면 "새 상대 여러 명과 거래했다"만 남는다.
  **주의**: 데이터가 시작한 직후에는 모든 쌍이 첫 거래로 보인다(1절에서 날짜별 비율을 본다). HI-Small 은 데이터가 10일뿐이라 보정 구간 앞쪽 며칠이 이 영향을 받는다.

- `환전 직후 송금`(2026-11-23 추가): 남에게 보낸 거래인데, **같은 계좌가 같은 시각(분)에 자기 계좌 안에서 환전**했고, 환전해 받은 통화가 보낸 통화와 같고,
  보낸 금액 ÷ 환전해 받은 금액이 `FX_RATIO`(0.8 ~ 1.25) 안인 거래다. 자기 계좌 안 환전은 송금 계좌 = 수취 계좌이고 보낸 통화 ≠ 받은 통화인 거래다
  - HI-Small(09-01 ~ 09-10)에서 이런 송금은 정상 거래의 1.5%, 6종 패턴 세탁의 58 ~ 72% 다. 1절의 패턴별 표에서 데이터셋마다 다시 본다
  - 시각을 "같은 분"으로 둔 이유: HI-Small 에서 금액이 맞는 환전을 앞뒤 1일까지 넓혀 찾아도 전부 같은 분에 있었다
  - 금액 비를 두는 이유: FAN 은 환전한 금액을 그대로 보내지만, 경유 계좌가 있는 패턴(SCATTER-GATHER · CYCLE · RANDOM)은 환전한 금액보다 조금 적게 보내는 거래가 있다(HI-Small 에서 대부분 20% 이내)
  - 쓰는 방식은 둘이다. **AND**: 환전 직후 송금만 센다. **OR**: 첫 거래이거나 환전 직후 송금이면 센다. 금액 조건은 어느 쪽이든 AND 다
  - **주의**: 논문의 패턴 정의에는 환전이 없다. 이 차이는 IBM 생성기가 세탁 송금을 만드는 방식에서 나온 것일 수 있다. 7-1절에서 이 조건을 뺀 결과와 나란히 본다

**`LOOKAHEAD_DAYS` 에 대해.** 룰은 뒤 거래가 들어와 패턴이 완성되면 앞 거래까지 근거로 잡는다. 실제 운영에서도 며칠 뒤 분석에서 잡힌다.
이 값을 올리면 "며칠 뒤에라도 잡히는가"를 볼 수 있다. 다만 HI-Small 은 09-10 뒤로 정상 거래가 거의 없고 세탁 거래만 이어져서 올리면 안 된다(룰에 유리해진다). HI-Large 에서만 쓴다.

**HI-Medium 의 구간에 대해.** (2026-11-23)
- 평가 구간은 00번과 같다(09-08 ~ 09-16). 모델 점수가 그 구간에만 있어서 같은 거래로 비교하려면 맞춰야 하고, 01 · 02번과도 같은 구간이 된다
- 룰도 데이터가 시작한 직후에는 오탐이 많다. HI-Small 에서 FAN-OUT 룰의 정상 오탐 254건 중 211건이 첫날이었고, 정상 거래 중 "처음 거래하는 상대" 비율이 첫날 51% → 5일째부터 4% 안팎이었다. 룰이 보는 것(직전 며칠의 거래, 그 상대와 거래한 적이 있는지)이 아직 안 쌓여서다
- 보정 구간을 09-05 ~ 09-07 로 둔 이유: 평가 구간과 겹치지 않아야 하고(겹치면 평가 구간에 맞춘 조건이 된다), 맨 앞 며칠은 위와 같은 이유로 조건을 고르기에도 맞지 않다. 1절의 날짜별 첫 거래 비율 표에서 정상 거래의 비율이 가라앉는 날을 보고 시작일을 고친다
- 기간 후보는 7일까지다. 평가 구간 첫날 기준으로 쌓인 이력이 7일이라 그보다 길게 돌아볼 수 없다. 1절의 시도 길이 표에서 7일로 덮이는지 본다
- HI-Small 의 보정 구간(09-01 ~ 09-08)은 첫날부터라 이 영향을 받는다. 이미 돌린 결과와 맞추려고 그대로 뒀다

**HI-Large 로 돌릴 때.** `DATASET` 만 바꾸면 된다. 다만 한 번도 돌려 본 적이 없다.
- 기간 후보가 7 · 14 · 30일인 이유: HI-Large 는 패턴이 길다. 시도 하나의 첫 거래부터 마지막 거래까지가 중앙값으로 FAN-IN · FAN-OUT · CYCLE 29일, RANDOM 21일, SCATTER-GATHER 32일, GATHER-SCATTER 56일이다(`docs/eda/large_eda.md` 3.1). 1 · 3 · 7일로는 시도의 일부만 보인다. GATHER-SCATTER 는 30일로도 전부는 안 보인다
- 읽는 거래가 보정 구간 30일 전부터라 약 1억 건이다. HI-Small 의 20배가 넘는다. **전부 돌리면 몇 시간이 걸릴 수 있다.** 룰 표는 파일에 쌓이고(`FILE_DB`), 표마다 걸린 시간(초)이 나온다
- 처음에는 `FILTERS` 에서 `'없음'` 과 `'1천↑'` 두 줄을 주석으로 막고 돌려 보는 것이 빠르다. 거래의 대부분이 이 두 필터에 있어서 시간도 대부분 여기서 쓴다. 이때 "조건값 하나" 줄은 비어서 나온다
- 중간에 끊기면 `REUSE = True` 로 두고 처음부터 다시 실행한다. 이미 만든 룰 표는 건너뛴다
- 경로형은 경로 수 상한이 `MAX_WALKS` 를 넘는 (필터, 기간)을 건너뛴다. 건너뛴 것과 만들지 않은 것은 5절 표에서 빠지고, 남은 것 중에서 조건을 고른다
- 작업 파일 `rule_engine_work.duckdb` 는 수십 GB 가 될 수 있다. 다 쓰면 지운다 (`.gitignore` 에 `*.duckdb` 가 있어 커밋되지는 않는다)

**금액 비 0.8 ~ 1.25 에 대해.** HI-Small 패턴 파일에서 CYCLE·RANDOM 시도의 이어지는 거래 금액 비(USD 환산)는 98% 가 이 범위 안이다(2026-11-23 확인).
통화가 계속 바뀌어도 USD 로 바꾸면 거의 유지된다. 이 조건이 없으면 경로 수가 폭발해서 원장 전체에 경로형 룰을 걸 수 없다.

## 1. 거래 읽기
보정 구간 앞 `PAD_DAYS` 일부터 평가 구간 끝까지를 DuckDB 표로 올린다. 금액은 서비스와 같은 고정 환율로 USD 로 바꾼다.

In [ ]:
import time
import duckdb, numpy as np, pandas as pd
import rule_lib as RL

TAGS = {name: RL.filter_tag(*spec) for name, spec in FILTERS.items()}     # 필터 이름 → 표 이름에 붙는 꼬리표 (예: m1000f, 환전 AND 는 m1000fx, 환전 OR 는 m1000fo)

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'parquets').is_dir()), None)
assert ROOT is not None, 'parquets/ 폴더를 못 찾음. 데이터가 있는 GPU 서버의 저장소 안에서 열어야 한다'
BASE = ROOT / 'parquets' / DATASET
OUT = Path.cwd() / 'results' / DATASET / 'rule_vs_model'
OUT.mkdir(parents=True, exist_ok=True)
START = str((pd.Timestamp(CALIB_WINDOW[0]) - pd.Timedelta(days=PAD_DAYS)).date())
END = str((pd.Timestamp(TEST_WINDOW[1]) + pd.Timedelta(days=LOOKAHEAD_DAYS)).date())
SPANS = {'보정': CALIB_WINDOW, '평가': TEST_WINDOW}
WORK_DB = BASE / 'rule_engine_work.duckdb' if FILE_DB else None
con = duckdb.connect(str(WORK_DB)) if FILE_DB else duckdb.connect()
con.execute(f'SET threads={THREADS}'); con.execute(f"SET memory_limit='{MEMORY}'")
if FILE_DB:
    con.execute(f"SET temp_directory='{BASE / 'rule_engine_tmp'}'")     # 메모리에 안 들어가는 중간 결과를 쓰는 곳
RL.REUSE = bool(REUSE and FILE_DB)
print(BASE, '| 읽는 기간', START, '~', END, '| 보정', CALIB_WINDOW, '| 평가', TEST_WINDOW, '| duckdb', duckdb.__version__, '| 작업 DB', WORK_DB, '| 이어 돌리기', RL.REUSE)

In [ ]:
RL.load_tx(con, BASE, START, END, GL.UNITS_PER_USD, EXCLUDE, FX_RATIO)     # 환율_모르는_거래 는 0 이어야 정상

In [ ]:
# 구간별 세탁 거래 수 (패턴별). 평가 구간의 수가 작으면 비율이 크게 흔들린다. 지표에서_뺌 = True 인 줄은 뒤의 지표에 들어가지 않는다
IN_CALIB = f"day BETWEEN DATE '{CALIB_WINDOW[0]}' AND DATE '{CALIB_WINDOW[1]}'"
IN_TEST = f"day BETWEEN DATE '{TEST_WINDOW[0]}' AND DATE '{TEST_WINDOW[1]}'"
con.execute(f'''SELECT CASE WHEN y = 1 THEN pattern_type ELSE '(정상)' END AS 패턴, bool_or(skip) AS 지표에서_뺌,
                       count(*) FILTER (WHERE {IN_CALIB}) AS 보정_거래, count(DISTINCT attempt_id) FILTER (WHERE {IN_CALIB}) AS 보정_시도,
                       count(*) FILTER (WHERE {IN_TEST}) AS 평가_거래, count(DISTINCT attempt_id) FILTER (WHERE {IN_TEST}) AS 평가_시도
                FROM tx_all GROUP BY 1 ORDER BY 1''').df()

In [ ]:
# 평가 구간에 걸친 시도 중, 구간이 끝난 뒤에도 거래가 이어지는 시도. 룰이 볼 수 있는 모양이 덜 완성돼 있다 (지표에서 빼는 패턴은 뺐다)
con.execute(f'''WITH a AS (SELECT attempt_id, any_value(pattern_type) AS 패턴, max(ts) AS last_ts,
                                 bool_or(CAST(ts AS DATE) BETWEEN DATE '{TEST_WINDOW[0]}' AND DATE '{TEST_WINDOW[1]}') AS touch
                          FROM read_parquet('{BASE / 'patterns.parquet'}') GROUP BY attempt_id)
                SELECT 패턴, count(*) AS 평가_시도, count(*) FILTER (WHERE CAST(last_ts AS DATE) > DATE '{TEST_WINDOW[1]}') AS 구간_뒤로_이어짐,
                       count(*) FILTER (WHERE CAST(last_ts AS DATE) > DATE '{END}') AS 룰이_끝을_못_봄
                FROM a WHERE touch AND 패턴 NOT IN (SELECT unnest({EXCLUDE!r}::VARCHAR[])) GROUP BY 1 ORDER BY 1''').df()

In [ ]:
# 패턴별 시도의 크기와 길이(첫 거래 ~ 마지막 거래). 기간 후보(WINDOWS)가 이 길이를 덮는지 본다. 'n일_이하' = 길이가 그 기간 안인 시도의 비율
cover = ', '.join(f'round(avg((d <= {W / 1440})::INT), 2) AS "{W // 1440}일_이하"' for W in WINDOWS)
con.execute(f'''WITH a AS (SELECT attempt_id, any_value(pattern_type) AS 패턴, count(*) AS n, date_diff('minute', min(ts), max(ts)) / 1440.0 AS d
                          FROM read_parquet('{BASE / 'patterns.parquet'}') GROUP BY attempt_id)
                SELECT 패턴, count(*) AS 시도, median(n) AS 거래_수_중앙값, max(n) AS 거래_수_최대,
                       round(median(d), 2) AS 길이_중앙값_일, round(quantile_cont(d, 0.9), 2) AS 길이_p90_일, round(max(d), 2) AS 길이_최대_일, {cover}
                FROM a WHERE 패턴 NOT IN (SELECT unnest({EXCLUDE!r}::VARCHAR[])) GROUP BY 1 ORDER BY 1''').df()

In [ ]:
# 날짜별 "첫 거래" 비율. 정상 거래의 비율이 높은 날은 '처음 거래하는 상대' 필터가 아직 제 구실을 못 하는 날이다(데이터 시작 직후)
con.execute('''SELECT day, count(*) AS 거래,
                      round(avg(first_pair::INT) FILTER (WHERE y = 0), 3) AS 정상_첫거래_비율,
                      round(avg(first_pair::INT) FILTER (WHERE y = 1 AND NOT skip), 3) AS 패턴_세탁_첫거래_비율
               FROM tx GROUP BY 1 ORDER BY 1''').df()

In [ ]:
# 패턴별 "환전 직후 송금" 비율 (읽은 기간 전체). 정상과 차이가 큰 패턴일수록 환전 조건이 듣는다. 그중_첫거래 = 환전 직후 송금이면서 첫 거래인 비율
con.execute('''SELECT CASE WHEN y = 1 THEN pattern_type ELSE '(정상)' END AS 패턴, bool_or(skip) AS 지표에서_뺌, count(*) AS 거래,
                      count(*) FILTER (WHERE fx_pair) AS 환전_직후_송금, round(avg(fx_pair::INT), 3) AS 비율,
                      round(avg((fx_pair AND first_pair)::INT), 3) AS 그중_첫거래,
                      round(avg(fx_pair::INT) FILTER (WHERE usd >= 1000), 3) AS 비율_1천_이상
               FROM tx GROUP BY 1 ORDER BY 1''').df()

## 2. 바쁜 계좌
거래가 몰리는 계좌(정산·수납 계좌 등)는 "상대 계좌가 많다"가 정보가 아니고, 경로가 그 계좌를 지나면 아무 거래나 이어진다.
그래서 바쁜 계좌는 **룰의 대상도 경유도 되지 않는다**(상대 계좌로는 센다). 룰엔진 설계안 3.6 을 단순하게 옮긴 것이다.
바쁜 계좌에 닿은 세탁 거래는 룰이 그 계좌 쪽에서는 못 잡는다. 그 수를 여기서 확인한다.

In [ ]:
RL.busy_accounts(con, BUSY_MAX, BUSY_MINUTES)

## 3. 거래 필터 (조건 ①)
필터마다 룰이 볼 거래 표를 따로 만든다. 필터를 세게 걸수록 정상 거래가 많이 빠지지만 세탁 거래도 같이 빠진다.
`그중_6종_패턴_세탁` 이 그 필터에서 룰이 잡을 수 있는 상한이다.

In [ ]:
pd.DataFrame([dict(필터=name, **RL.apply_filter(con, TAGS[name], *FILTERS[name])) for name in FILTERS])

## 4. 룰 돌리기 (조건 ② ③ ④)
모든 룰이 같은 방식으로 돈다. 필터를 통과한 거래만 본다.

1. **트리거**: 대상 계좌의 거래 하나하나를 검사 시점으로 삼는다.
2. **기간**: 트리거 시각을 포함해 직전 `W` 분을 돌아본다.
3. **세기**: 그 기간 안에서 패턴의 크기(계좌 수, 경로 길이)를 센다. 세기가 조건값 이상이면 적중이다.
4. **근거 거래**: 적중한 트리거의 기간 안에서 그 패턴을 이루는 거래. 이것이 룰이 "고른 거래"다.

거래마다 "자기를 근거로 삼는 트리거들의 세기 중 최댓값"을 저장한다. 그러면 조건값을 바꿀 때 룰을 다시 돌리지 않고 `세기 ≥ 조건값` 으로 걸러 보기만 하면 된다.
아래 셀들은 필터 × 기간마다 한 번씩 돈다.

### 4-1. FAN-OUT · FAN-IN (집계형)

| | FAN-OUT | FAN-IN |
|---|---|---|
| 모양 | A → 여러 계좌 | 여러 계좌 → A |
| 대상 | 송금 계좌 A | 수취 계좌 A |
| 트리거 | A 의 출금 | A 로 들어온 입금 |
| 세기 | 기간 안 **서로 다른 수취 계좌 수** | 기간 안 **서로 다른 송금 계좌 수** |
| 근거 거래 | 기간 안 A 의 출금 전부 | 기간 안 A 의 입금 전부 |

**반복 없음**(조건 ④): 기간 안 거래 수 = 상대 계좌 수. 즉 그 기간에 모든 상대와 **한 번씩만** 거래했다는 조건이다.
같은 상대에게 여러 번 보내는 계좌(월급·대금 지급 등)를 뺀다. 표에는 `v`(반복 허용)와 `v_once`(반복 없음) 두 세기가 같이 들어간다.

In [ ]:
pd.DataFrame([dict(필터=name, **RL.fan(con, side, W, TAGS[name])) for name in FILTERS for W in WINDOWS for side in ('out', 'in')])

### 4-2. GATHER-SCATTER (집계형)
여러 계좌 → H → 여러 계좌. 대상은 가운데 계좌 H, 트리거는 H 의 출금이다.

- `din` = 기간 안 입금의 서로 다른 송금 계좌 수
- `dout` = **기간 안 첫 입금 뒤** 출금의 서로 다른 수취 계좌 수 (모은 뒤에 뿌린 것만 센다)
- 세기 = `min(din, dout)`. 조건값 K 는 "양쪽 다 K 개 이상"이라는 뜻이다
- 근거 거래 = 기간 안 입금 전부 + 첫 입금 뒤의 출금

In [ ]:
pd.DataFrame([dict(필터=name, **RL.gather_scatter(con, W, TAGS[name])) for name in FILTERS for W in WINDOWS])

### 4-3. SCATTER-GATHER (2홉형)
S → 경유 계좌 여러 개 → D. 대상은 도착 계좌 D, 트리거는 D 로 들어온 입금(경유 → D)이다.

- 세기 = 같은 출발 계좌 S 에서 와서 D 에 닿은 **경유 계좌 수**. 경유 계좌마다 `S → 경유` 가 `경유 → D` 보다 먼저(같은 분 포함)여야 하고 둘 다 기간 안이어야 한다
- 근거 거래 = 그 경유 계좌들의 `S → 경유` 거래와 `경유 → D` 거래
- S = D 는 순환이라 뺀다

먼저 거래 시각과 무관하게 "경유 계좌를 2개 이상 공유하는 (S, D)"만 추리고, 그 안에서만 시각을 본다.

In [ ]:
pd.DataFrame([dict(필터=name, **RL.sg_candidates(con, TAGS[name])) for name in FILTERS])

In [ ]:
pd.DataFrame([dict(필터=name, **RL.scatter_gather(con, W, TAGS[name])) for name in FILTERS for W in WINDOWS])

### 4-4. CYCLE · RANDOM (경로형)
거래를 하나씩 이어 경로를 만든다. 두 거래가 **이어진다**는 것은 다음을 모두 만족한다는 뜻이다.

- 앞 거래의 수취 계좌 = 뒤 거래의 송금 계좌
- 뒤 거래가 앞 거래 이후(같은 분 포함)이고 기간 `W` 안
- 뒤 금액 ÷ 앞 금액(USD)이 `RATIO_MIN ~ RATIO_MAX`

경로는 같은 계좌를 두 번 지나지 않고, 첫 거래부터 마지막 거래까지가 `W` 안이며, 거래 수는 `MAX_HOPS` 까지다.

| | CYCLE | RANDOM |
|---|---|---|
| 모양 | A → B → … → A | A → B → C → … (돌아오지 않음) |
| 적중 | 경로가 **출발 계좌로 돌아옴** (거래 2건 이상) | 돌아오지 않은 경로 |
| 세기 | 그 거래가 든 순환 중 가장 긴 것의 거래 수 | 그 거래가 든 경로 중 가장 긴 것의 거래 수 |
| 근거 거래 | 순환의 거래 전부 | 경로의 거래 전부 |

순환의 앞부분도 닫히기 전까지는 "돌아오지 않은 경로"라서 RANDOM 에도 걸린다.

In [ ]:
pd.DataFrame([dict(필터=name, **RL.links(con, W, TAGS[name], RATIO_MIN, RATIO_MAX)) for name in FILTERS for W in PATH_WINDOWS])

In [ ]:
# 경로를 만들기 전에 길이(거래 수)별 경로 수의 상한을 본다. 합이 MAX_WALKS 를 넘는 (필터, 기간)은 다음 셀에서 건너뛴다
walk_bound = {(name, W): RL.walk_counts(con, W, TAGS[name], MAX_HOPS) for name in FILTERS for W in PATH_WINDOWS}
pd.concat([pd.DataFrame(walk_bound), pd.DataFrame({key: {'합': sum(v.values())} for key, v in walk_bound.items()})]).rename_axis('거래 수')

In [ ]:
pd.DataFrame([dict(필터=name, 기간_분=W, **RL.paths(con, W, TAGS[name], MAX_HOPS, sum(walk_bound[(name, W)].values()), MAX_WALKS))
              for name in FILTERS for W in PATH_WINDOWS])

## 5. 조건을 바꾸면 (보정 구간)
룰 × 필터 × 반복 × 기간 × 조건값을 전부 걸어 본다. 아래는 룰마다 보정 구간 F1 이 높은 조건 순이다.

- `대상_계좌`: 근거 거래가 걸린 서로 다른 대상 계좌 수. 사람이 봐야 할 양이다
- `고른_거래`: 룰이 고른 거래(근거 거래) 수
- `정밀도`: 고른 거래 중 세탁 거래 비율. 패턴 종류는 따지지 않는다 (FAN-OUT 룰이 GATHER-SCATTER 의 뿌리는 쪽을 잡아도 맞은 것으로 센다). 지표에서 빼는 거래는 고른 거래에도 넣지 않는다
- `재현율`: **그 룰이 노리는 패턴**의 세탁 거래 중 고른 비율
- 조건값 × 기간 모양으로 보려면 `RL.view(sweep_calib, 'fan_out', '1천↑', '없음')` 처럼 부른다 (룰, 필터, 반복)

In [ ]:
sweep_all = RL.sweep(con, GRID, WINDOWS, TAGS, SPANS)
sweep_calib = sweep_all[sweep_all.구간 == '보정'].drop(columns='구간').reset_index(drop=True)
sweep_test = sweep_all[sweep_all.구간 == '평가'].drop(columns='구간').reset_index(drop=True)
print(f'조건 {len(sweep_calib):,} 벌')
RL.top(sweep_calib, 'fan_out')

In [ ]:
RL.top(sweep_calib, 'fan_in')

In [ ]:
RL.top(sweep_calib, 'gather_scatter')

In [ ]:
RL.top(sweep_calib, 'scatter_gather')

In [ ]:
RL.top(sweep_calib, 'cycle')

In [ ]:
RL.top(sweep_calib, 'random')

## 6. 조건 고르기
룰마다 보정 구간에서 `PICK_BY`(기본 F1 = 위 정밀도와 재현율의 조화평균)가 가장 큰 조건 한 벌을 고른다.
모델의 의심 기준도 검증 구간에서 F1 이 가장 큰 값으로 고른 것이라 같은 기준이다.
다른 조건을 쓰려면 아래 셀에서 `CHOSEN` 을 직접 바꾼다. 룰을 빼려면 `CHOSEN.pop('fan_in')` 처럼 지운다.

In [ ]:
CHOSEN = RL.pick(sweep_calib, PICK_BY)       # {룰: dict(필터, 반복, 기간_분, 조건값)}
# CHOSEN['fan_in'] = dict(필터='첫거래+5천↑', 반복='허용', 기간_분=10080, 조건값=8)      # 직접 정할 때
CHOSEN

In [ ]:
# AND 로 묶은 효과: "조건값 하나만"(필터 없음, 반복 허용) 중 가장 좋은 것과 나란히 본다. 보정 구간
SINGLE = RL.pick(sweep_calib[(sweep_calib.필터 == '없음') & (sweep_calib.반복 != '없음')], PICK_BY)
SHOW = RL.KEYS + ['대상_계좌', '고른_거래', '세탁', '정밀도', '재현율', 'F1']
effect = pd.concat([RL.chosen_rows(sweep_calib, SINGLE)[SHOW].assign(방식='조건값 하나'),
                    RL.chosen_rows(sweep_calib, CHOSEN)[SHOW].assign(방식='AND 로 묶음')])
effect = effect.set_index(['룰', '방식']).loc[[RL.RULES[rule] for rule in CHOSEN]]
effect

## 7. 평가 구간 — 룰만
고른 조건을 평가 구간에 그대로 건다. 여기부터는 조건을 바꾸지 않는다.

In [ ]:
rule_table = pd.concat([RL.chosen_rows(sweep_test, SINGLE)[SHOW].assign(방식='조건값 하나'),
                        RL.chosen_rows(sweep_test, CHOSEN)[SHOW].assign(방식='AND 로 묶음')])
rule_table = rule_table.set_index(['룰', '방식']).loc[[RL.RULES[rule] for rule in CHOSEN]]
rule_table

In [ ]:
# 고른 조건에서 거래마다 룰에 걸렸는지를 가져온다 (보정 + 평가 구간). calib · test 는 지표에서 빼는 거래를 뺀 구간이다
F = RL.collect(con, CHOSEN, TAGS, CALIB_WINDOW[0], TEST_WINDOW[1])
calib, test = RL.day_mask(F, CALIB_WINDOW), RL.day_mask(F, TEST_WINDOW)
each = {RL.RULES[rule]: F[RL.RULES[rule]].to_numpy() for rule in CHOSEN}
rule_hit = np.logical_or.reduce(list(each.values()))          # 룰 중 하나라도 고른 거래
F1 = RL.collect(con, SINGLE, TAGS, CALIB_WINDOW[0], TEST_WINDOW[1])
single_hit = F1[[RL.RULES[rule] for rule in SINGLE]].to_numpy().any(axis=1)   # "조건값 하나" 룰들의 합집합 (참고)
del F1
print(f'{len(F):,} 건 | 보정 {calib.sum():,} | 평가 {test.sum():,} | 지표에서 뺀 거래 {int(F.skip.sum()):,} | 룰이 고른 거래 {int(rule_hit.sum()):,}')

In [ ]:
# 어떤 룰이 어떤 패턴의 세탁 거래를 잡았나. 값 = 그 패턴의 세탁 거래 중 그 룰이 고른 비율
cross = RL.by_pattern(F, test, {**each, '룰_합집합': rule_hit})
cross

In [ ]:
# 룰을 합친 결과. 보정 구간은 조건을 고른 구간이라 낙관적이지만, 룰이 뒤 거래까지 본 상태의 값이다
union_table = pd.DataFrame([RL.method_metrics('조건값 하나 · 보정 구간', single_hit, F, calib),
                            RL.method_metrics('AND 로 묶음 · 보정 구간', rule_hit, F, calib),
                            RL.method_metrics('조건값 하나 · 평가 구간', single_hit, F, test),
                            RL.method_metrics('AND 로 묶음 · 평가 구간', rule_hit, F, test)])
union_table

In [ ]:
# 룰 하나를 빼면 합집합이 어떻게 되나 (평가 구간). 빼자마자 정밀도가 크게 오르는 룰이 합집합을 흐리는 룰이다
drop_table = pd.DataFrame([RL.method_metrics(f'{name} 빼고', np.logical_or.reduce([m for other, m in each.items() if other != name]), F, test)
                           for name in each])
drop_table

### 7-1. 환전 조건을 넣으면 — AND 와 OR
환전 직후 송금(0절 설명) 조건을 쓰는 방식별로 조건을 **따로 골라서** 나란히 본다. 고르는 방법은 6절과 같다(보정 구간 `PICK_BY`).

| 방식 | 뜻 |
|---|---|
| 기존 조건 | 환전 조건을 쓰지 않는 필터 후보 안에서 고른 것 |
| 환전 AND | 환전 직후 송금만 세는 필터 후보 안에서 고른 것 |
| 필터 안 OR | "첫 거래이거나 환전 직후 송금"을 세는 필터 후보 안에서 고른 것 |
| 룰 단위 OR | `기존 조건` 으로 고른 거래 ∪ `환전 AND` 로 고른 거래. 룰을 두 벌 두고 어느 쪽이든 걸리면 고른다 |

6절의 `CHOSEN` 은 후보 전체에서 고른 것이라 앞의 세 방식 중 보정 구간 값이 가장 큰 것과 같다. `룰 단위 OR` 는 조건 한 벌이 아니라서 `CHOSEN` 에 들어가지 않는다.

In [ ]:
# 방식마다 그 방식의 필터 후보 안에서만 조건을 고른다
FX_GROUPS = {'기존 조건': [n for n, spec in FILTERS.items() if len(spec) < 3 or spec[2] is None],
             '환전 AND': [n for n, spec in FILTERS.items() if len(spec) == 3 and spec[2] == 'and'],
             '필터 안 OR': [n for n, spec in FILTERS.items() if len(spec) == 3 and spec[2] == 'or']}
BY_GROUP = {group: RL.pick(sweep_calib[sweep_calib.필터.isin(names)], PICK_BY) for group, names in FX_GROUPS.items() if names}
pd.DataFrame({group: {RL.RULES[rule]: f"{c['필터']} · 반복 {c['반복']} · {c['기간_분'] // 1440}일 · {c['조건값']} 이상" for rule, c in chosen.items()}
              for group, chosen in BY_GROUP.items()})

In [ ]:
# 방식마다 고른 조건에서 거래별 적중을 가져온다. 줄 순서는 F 와 같다
never = np.zeros(len(F), dtype=bool)
fx_hits = {}
for group, chosen in BY_GROUP.items():
    G = RL.collect(con, chosen, TAGS, CALIB_WINDOW[0], TEST_WINDOW[1])
    assert (G.edge_id.to_numpy() == F.edge_id.to_numpy()).all()
    fx_hits[group] = {label: (G[label].to_numpy() if label in G else never) for label in each}
del G
if '기존 조건' in fx_hits and '환전 AND' in fx_hits:      # 룰 단위 OR: 두 벌 중 어느 쪽이든 걸린 거래
    fx_hits['룰 단위 OR'] = {label: fx_hits['기존 조건'][label] | fx_hits['환전 AND'][label] for label in each}
{group: {label: int(mask.sum()) for label, mask in masks.items()} for group, masks in fx_hits.items()}

In [ ]:
# 룰별 결과. 보정 구간은 조건을 고른 구간이라 낙관적이다. 평가 구간 값으로 비교한다
fx_table = pd.concat({span: pd.concat([RL.rule_metrics(F, window, masks).assign(방식=group) for group, masks in fx_hits.items()]).set_index(['룰', '방식'])
                      for span, window in (('보정', calib), ('평가', test))}, axis=1).loc[list(each)]
fx_table

In [ ]:
# 룰을 합친 결과 (평가 구간). 맨 아래 줄은 룰 없이 "환전 직후 송금" 전부를 고른 것 (참고)
fx_any = F.fx_pair.to_numpy(dtype=bool)
fx_union = pd.DataFrame([RL.method_metrics(group, np.logical_or.reduce(list(masks.values())), F, test) for group, masks in fx_hits.items()]
                        + [RL.method_metrics('환전 직후 송금 전부 (룰 없이)', fx_any, F, test)])
fx_union

## 8. 평가 구간 — 이진분류만

In [ ]:
if SCORES is None:
    model_hit = None
    print('MODEL 을 정하지 않아 모델 쪽은 건너뛴다. 0절에서 MODEL 을 정하고 처음부터 다시 돌린다')
else:
    scores = pd.read_parquet(SCORES)[['edge_id', 'y', 'score']].drop_duplicates('edge_id')     # edge_id, y, score
    joined = F[['edge_id', 'y']].merge(scores, on='edge_id', how='left', suffixes=('', '_score'))
    assert len(joined) == len(F)
    score = joined.score.to_numpy(dtype='float64')
    print(f'점수 {len(scores):,} 건 | 평가 구간인데 점수가 없는 거래 {int((test & np.isnan(score)).sum()):,} (0 이어야 정상)',
          '| 라벨 불일치', int((joined.y_score.notna() & (joined.y_score != joined.y)).sum()))
    model_hit = test & (np.nan_to_num(score, nan=-1.0) >= THRESHOLD)
    print('의심 기준', THRESHOLD, '| 모델이 고른 거래', int(model_hit.sum()))

## 9. 비교
### 9-1. 거래 단위
- `재현율`: 평가 구간의 세탁 거래 중 고른 비율. `EXCLUDE` 의 세탁은 분모에도 고른 거래에도 없다. 기본 설정에서는 룰이 있는 6종 패턴의 세탁이 분모다
- `EXCLUDE` 를 비우면 분모를 좁힌 `재현율_패턴_전체`(무패턴 제외), `재현율_룰이_있는_6종` 열이 같이 나온다
- `이진분류 상위 N건`: 룰이 고른 건수와 **같은 건수**를 모델 점수 높은 순으로 고른 것. 같은 검토량에서의 비교다

In [ ]:
methods = {'룰 합집합': rule_hit & test}
single = {f'룰 · {name}': mask & test for name, mask in each.items()}      # 룰 하나씩 (참고 줄)
if model_hit is not None:
    n_rule = int((rule_hit & test).sum())
    order = np.argsort(-np.where(test, np.nan_to_num(score, nan=-1.0), -np.inf), kind='stable')
    top = np.zeros(len(F), dtype=bool)
    top[order[:n_rule]] = True
    methods = {'이진분류만': model_hit, **methods, f'이진분류 상위 {n_rule:,}건': top & test,
               '둘 다 고름': model_hit & rule_hit, '둘 중 하나라도': model_hit | (rule_hit & test)}
compare = pd.DataFrame([RL.method_metrics(name, mask, F, test) for name, mask in {**methods, **single}.items()])
compare

### 9-2. 패턴별 재현율

In [ ]:
pattern_table = RL.by_pattern(F, test, methods)
pattern_table

### 9-3. 세탁 거래를 누가 잡았나
평가 구간의 세탁 거래를 패턴별로 '둘 다 / 모델만 / 룰만 / 둘 다 놓침' 으로 나눈다. `룰만` 이 룰이 모델에 더해 주는 몫이다.

In [ ]:
if model_hit is None:
    who_table = None
    print('모델 점수가 없어 건너뛴다')
else:
    y, kind = F.y.to_numpy(), F.pattern_type.to_numpy()
    who = np.select([model_hit & rule_hit, model_hit, rule_hit], ['둘 다', '모델만', '룰만'], '둘 다 놓침')
    base = test & (y == 1)
    who_table = pd.crosstab(pd.Series(kind[base], name='패턴'), pd.Series(who[base], name='잡은 쪽'), margins=True)
who_table

### 9-4. 세탁 시도 단위
시도(Patterns 파일의 BEGIN ~ END 한 덩어리)마다 평가 구간 안 거래 중 몇 건을 골랐는지 본다. 값은 그런 시도의 비율이다.

In [ ]:
main = {name: mask for name, mask in methods.items() if name in ('이진분류만', '룰 합집합')}
attempt_table = RL.by_attempt(F, test, main)
attempt_table

### 9-5. 평가 구간 뒤로 이어지는 시도 vs 구간 안에 끝난 시도
평가 구간이 끝난 뒤에도 거래가 이어지는 시도는 구간 안에서 **완성되기 전 모습**만 보인다. 룰은 패턴이 완성돼야 걸리므로 특히 불리하다. 01 · 02번과 같은 기준으로 나눈다.
- `구간 안에 끝남`: 그 시도의 마지막 거래가 평가 구간 마지막 날 이전
- `구간 뒤로 이어짐`: 마지막 거래가 그 뒤

값은 그 칸의 세탁 거래 중 고른 비율이다.

In [ ]:
tail = GL.attempt_tail(con, BASE / 'patterns.parquet', TEST_WINDOW[1]).set_index('edge_id')
after = F.edge_id.map(tail.after).fillna(False).astype(bool).to_numpy()      # 그 거래의 시도가 평가 구간 뒤로 이어지는가
tail_table = RL.by_tail(F, test, main, after, GL.TAIL_LABELS)
tail_table

### 9-6. 룰별 결과를 시도의 끝으로 나눠서
7절의 룰별 표는 평가 구간 뒤로 이어지는 시도의 거래까지 한데 넣은 값이다. 여기서는 둘로 나눈다. 방식은 7-1절의 것이다(`기존 조건` = 환전 조건 없이 고른 룰).
- `구간 안에 끝남`: 뒤로 이어지는 시도의 세탁 거래를 **정답도 오답도 아닌 것으로 빼고** 센다. 정상 거래는 그대로 둔다
- `구간 뒤로 이어짐`: 그 시도들의 세탁 거래만 센다. 정상 거래가 없어 정밀도는 뜻이 없다. `패턴_세탁` · `패턴_고름` · `재현율` 만 본다

In [ ]:
tail_rule = pd.concat([RL.rule_metrics(F, test & (after == flag), masks).assign(방식=group, 끝=label)
                       for group, masks in fx_hits.items() for flag, label in GL.TAIL_LABELS.items()]).set_index(['방식', '끝', '룰'])
tail_rule.loc['기존 조건']

In [ ]:
# 룰을 합친 결과도 같은 방식으로 나눈다. 맨 위 방식은 6절에서 고른 조건(CHOSEN) 그대로다
tail_union = pd.DataFrame([dict(끝=label, **RL.method_metrics(group, np.logical_or.reduce(list(masks.values())), F, test & (after == flag)))
                           for group, masks in {'고른 조건 (6절)': each, **fx_hits}.items() for flag, label in GL.TAIL_LABELS.items()])
tail_union

In [ ]:
# 어떤 룰이 어떤 패턴의 세탁 거래를 잡았나 (7절의 cross)도 시도의 끝으로 나눈다. 값 = 그 칸의 세탁 거래 중 그 룰이 고른 비율
tail_cross = pd.concat([RL.by_pattern(F, test & (after == flag), {**masks, '룰_합집합': np.logical_or.reduce(list(masks.values()))}).assign(방식=group, 끝=label)
                        for group, masks in {'고른 조건 (6절)': each, **fx_hits}.items() for flag, label in GL.TAIL_LABELS.items()]).set_index(['방식', '끝', '패턴'])
tail_cross.loc['기존 조건']

### 9-7. 룰 적중 묶음 (05번에서 읽는다)
룰이 고른 거래를 **같은 대상 때문에 고른 것끼리** 나눈다. 05번이 이 묶음으로 생성기 없이 Alert 를 만든다.
- FAN-OUT · FAN-IN · GATHER-SCATTER: 대상 계좌(뿌리는 / 모으는 / 가운데 계좌)가 같은 근거 거래
- SCATTER-GATHER: 도착 계좌가 같은 근거 거래
- 위 넷은 대상이 같아도, 이웃한 두 근거 거래의 간격이 그 룰의 기간을 넘으면 거기서 나눈다
- CYCLE · RANDOM: 근거 거래끼리 "이어지는 쌍"으로 연결된 덩어리
- 읽은 기간 전체에서 만든다. 평가 구간만 고르는 것은 05번이 한다

In [ ]:
groups = RL.hit_groups(con, CHOSEN, TAGS, START, END)
# 확인: 7절에서 쓴 구간 안에서는 (거래, 룰) 이 룰이 고른 거래와 정확히 같아야 한다
inside = groups[groups.edge_id.isin(F.edge_id)]
for label, mask in each.items():
    assert set(inside.edge_id[inside.룰 == label]) == set(F.edge_id[mask]), label
size = groups.groupby(['룰', '묶음']).size()
size.groupby('룰').agg(묶음='size', 거래='sum', 크기_중앙값='median', 크기_최대='max', 거래_1건짜리=lambda s: int((s == 1).sum())).loc[list(each)]

## 10. 저장

In [ ]:
sweep_calib.to_csv(OUT / 'sweep_calib.csv', index=False)
sweep_test.to_csv(OUT / 'sweep_test.csv', index=False)
effect.to_csv(OUT / 'rules_calib.csv')
rule_table.to_csv(OUT / 'rules_test.csv')
cross.to_csv(OUT / 'rule_by_pattern.csv', index=False)
union_table.to_csv(OUT / 'rule_union.csv', index=False)
drop_table.to_csv(OUT / 'rule_drop_one.csv', index=False)
fx_table.to_csv(OUT / 'fx_by_rule.csv')
fx_union.to_csv(OUT / 'fx_union.csv', index=False)
compare.to_csv(OUT / 'compare.csv', index=False)
pattern_table.to_csv(OUT / 'by_pattern.csv', index=False)
attempt_table.to_csv(OUT / 'by_attempt.csv', index=False)
tail_table.to_csv(OUT / 'by_tail.csv', index=False)
tail_rule.to_csv(OUT / 'by_tail_rule.csv')
tail_union.to_csv(OUT / 'by_tail_union.csv', index=False)
tail_cross.to_csv(OUT / 'by_tail_cross.csv')
if who_table is not None:
    who_table.to_csv(OUT / 'who.csv')
# 고른 거래 목록. 룰 씨앗을 생성기에 넣는 다음 실험에서 읽는다
model_any = None if model_hit is None else np.nan_to_num(score, nan=-1.0) >= THRESHOLD      # 지표에서 빼는 거래까지 포함한 모델 판정
keep = rule_hit | (F.y.to_numpy() == 1) | (model_any if model_any is not None else False)     # skip 열로 지표에서 뺀 거래를 구분한다
hits = F[keep].copy()
if model_any is not None:
    hits['score'], hits['model'] = score[keep], model_any[keep]
hits.to_parquet(OUT / 'hits.parquet', index=False)
groups.to_parquet(OUT / 'hit_groups.parquet', index=False)      # 룰 적중 묶음. 05번이 읽는다
json.dump(dict(dataset=DATASET, model=MODEL, scores=None if SCORES is None else str(SCORES), threshold=THRESHOLD,
               test_window=TEST_WINDOW, calib_window=CALIB_WINDOW, pad_days=PAD_DAYS, lookahead_days=LOOKAHEAD_DAYS, exclude=EXCLUDE,
               filters=FILTERS, windows=WINDOWS, path_windows=PATH_WINDOWS, max_walks=MAX_WALKS, skipped=sorted(RL.skipped(con)), grid=GRID, ratio=[RATIO_MIN, RATIO_MAX], max_hops=MAX_HOPS,
               busy=[BUSY_MAX, BUSY_MINUTES], pick_by=PICK_BY, chosen=CHOSEN, single=SINGLE, fx_ratio=FX_RATIO, chosen_by_fx=BY_GROUP, duckdb=duckdb.__version__),
          open(OUT / 'run.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(sorted(p.name for p in OUT.iterdir()))